# Agent Memory Server - 25 Minute Interactive Demo

This notebook focuses on: setup, code-driven memory, working memory, background extraction, search, and one episodic-memory example.


## Quick Start

Before running this notebook, make sure you have:

1. **Redis running**
2. **AMS running**
3. **`OPENAI_API_KEY` set**
4. **Python dependencies installed** (`agent-memory-client`, `openai`, `python-dotenv`)

This is designed for roughly **20-25 minutes live**.


You can get a local Redis instance running, as well as a local Agent Memory Server by running the included docker `compose.yaml` file.
```
docker compose up -d
```

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import required libraries
import os
from datetime import datetime, timezone
from dotenv import load_dotenv


# Load environment variables from .env file
load_dotenv()

# Configuration
BASE_URL = "http://localhost:8000"
NAMESPACE = "travel_agent"

print(f"Base URL: {BASE_URL}")
print(f"Namespace: {NAMESPACE}")
print(f"OpenAI API Key: {'loaded' if os.environ.get('OPENAI_API_KEY') else '✗ not found'}")

Base URL: http://localhost:8000
Namespace: travel_agent
OpenAI API Key: loaded


In [3]:
# Initialize the Python SDK client
from agent_memory_client import MemoryAPIClient, MemoryClientConfig

config = MemoryClientConfig(
    base_url=BASE_URL,
    timeout=30.0,
    default_namespace=NAMESPACE,  # "travel_agent"
)

client = MemoryAPIClient(config)
print("Memory client initialized!")
print("health check (current time):",await client.health_check())
print(f"Default namespace: {config.default_namespace}")

Memory client initialized!
health check (current time): now=1774983959794.0
Default namespace: travel_agent


## Demo Agenda

This notebook covers:

1. Code-driven memory with `memory_prompt()`
2. Writing and inspecting working memory
3. Background extraction
4. Long-term search
5. episodic-memory


# Pattern 1: Code-Driven Memory

Manually store and retrieve memories about a user session.\
Use this when your application code should decide exactly when to read or write memory.

### Step 1: Initialize memory client

In [4]:
# Pattern 1 Code driven

from agent_memory_client.models import (
    ClientMemoryRecord,
    MemoryMessage,
    MemoryTypeEnum,
    WorkingMemory,
)


# Use "nitin" as our demo user
SESSION_ID = "nitin-travel-session"
USER_ID = "nitin"

# Step 1: Create/get a working memory session for Nitin
created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID, namespace=NAMESPACE, user_id=USER_ID
)

print(f"Session {'created' if created else 'already existed'}: {SESSION_ID}")
print(f"User: {USER_ID}")
print(f"Namespace: {NAMESPACE}")

Session created: nitin-travel-session
User: nitin
Namespace: travel_agent


### Step 2: Seed data into longterm memory

In [5]:
# Nitin's travel preferences (from the travel agent demo)
memories_to_store = [
    ClientMemoryRecord(
        text="Comfort travel - middle tier (not luxury, not budget)",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Vegetarian diet - needs vegetarian restaurant options",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Extra leg room on flights (premium economy or exit row and budget allows, anything goes)",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Prefers hotels with good amenities",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
    ClientMemoryRecord(
        text="Enjoys technology, sports, outdoors, and innovation hubs",
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=["travel", "preferences"],
        user_id=USER_ID,
        namespace=NAMESPACE,
    ),
]

print(f"Storing {len(memories_to_store)} preferences for user '{USER_ID}'...")
result = await client.create_long_term_memory(memories_to_store)
print(f"Result: {result.status}")

# Show what we just stored
print("\nStored memories:")
for i, mem in enumerate(memories_to_store, 1):
    print(f"  {i}. {mem.text}")

Storing 5 preferences for user 'nitin'...
Result: ok

Stored memories:
  1. Comfort travel - middle tier (not luxury, not budget)
  2. Vegetarian diet - needs vegetarian restaurant options
  3. Extra leg room on flights (premium economy or exit row and budget allows, anything goes)
  4. Prefers hotels with good amenities
  5. Enjoys technology, sports, outdoors, and innovation hubs


### Step 3: Retrieve relevant context

`client.search_long_term_memory(...)` searches our stored memories for knowledge similar to the `text` parameter provided. The minimum level of similarity is controlled by the `distance_threshold` parameter.

`client.memory_prompt(...)` performs a similar search, and formats the memories to be ready to be passed to an LLM prompt.

   

In [6]:
# Wait for memories to be indexed (embedding + Redis indexing happens asyncronously)
result = await client.search_long_term_memory(
    text="travel preferences",
    limit=5,
    user_id={"eq": USER_ID},
    namespace={"eq": NAMESPACE},
    distance_threshold=0.65,
)

print("Memories associated with the user's travel preferences are:")
for memory in result.memories:
    print(memory)

Memories associated with the user's travel preferences are:
id='01KN01PF648TR7SHJJRZK1QZYV' text='Comfort travel - middle tier (not luxury, not budget)' session_id=None user_id='nitin' namespace='travel_agent' last_accessed=datetime.datetime(2026, 3, 31, 19, 5, 59, tzinfo=TzInfo(0)) created_at=datetime.datetime(2026, 3, 30, 18, 57, 0, 100000, tzinfo=TzInfo(0)) updated_at=datetime.datetime(2026, 3, 30, 18, 57, 0, 100000, tzinfo=TzInfo(0)) topics=['travel', 'preferences'] entities=[] memory_hash='32702233bbeefd16bee2def9a3b2a8e39f4d82859dcfbf09e6770cd67eac5063' discrete_memory_extracted='f' memory_type=<MemoryTypeEnum.SEMANTIC: 'semantic'> persisted_at=None extracted_from=[] event_date=None dist=0.517619252205 score=0.48238074779499995 score_type=<SearchScoreTypeEnum.SEMANTIC: 'semantic'>
id='01KN01PF648TR7SHJJRZK1QZYX' text='Extra leg room on flights (premium economy or exit row and budget allows, anything goes)' session_id=None user_id='nitin' namespace='travel_agent' last_accessed=dat

In [7]:
# Perform a search and format the results for an LLM prompt
user_query = "What are my travel preferences?"

print(f"\nQuery: '{user_query}'")
print(f"Searching memories for user: {USER_ID}\n")

# Note: The memory_prompt() endpoint is designed to return ready-to-use messages for an LLM.
# The memories are pre-formatted into the system prompt so you can directly
# pass messages to OpenAI/Claude without additional processing.
prompt_result = await client.memory_prompt(
    session_id=SESSION_ID,
    query=user_query,
    namespace=NAMESPACE,
    user_id=USER_ID,
    long_term_search={
        "limit": 5,
        # distance_threshold: Lower = stricter when set. If omitted, the server
        # uses no distance filter (distance_threshold=None) for broader KNN recall.
        "user_id": {"eq": USER_ID}  # Only search Nitin's memories
    }
)

messages = prompt_result.get("messages", [])

# Count long-term memories from the structured response
# The memory_prompt() API returns both formatted messages AND a long_term_memories list
long_term_memories = prompt_result.get("long_term_memories") or []
memory_count = len(long_term_memories)

print(f"Long-term memories found: {memory_count}\n")

# Display the messages
for msg in messages:
    role = msg.get("role", "unknown")
    content = msg.get("content", {})
    if isinstance(content, dict):
        text = content.get("text", str(content))
    else:
        text = str(content)
    print(f"[{role.upper()}]")
    print(f"{text}\n")


Query: 'What are my travel preferences?'
Searching memories for user: nitin

Long-term memories found: 5

[SYSTEM]
## Long term memories related to the user's query
 - Comfort travel - middle tier (not luxury, not budget) (ID: 01KN01PF648TR7SHJJRZK1QZYV)
- Prefers hotels with good amenities (ID: 01KN01PF648TR7SHJJRZK1QZYY)
- Extra leg room on flights (premium economy or exit row and budget allows, anything goes) (ID: 01KN01PF648TR7SHJJRZK1QZYX)
- Enjoys technology, sports, outdoors, and innovation hubs (ID: 01KN01PF648TR7SHJJRZK1QZYZ)
- Vegetarian diet - needs vegetarian restaurant options (ID: 01KN01PF648TR7SHJJRZK1QZYW)

[USER]
What are my travel preferences?



### Step 4: Store conversation in working memory

Working memory stores the current conversation for this session.
This is separate from long-term memory (which stores facts/preferences).

In [8]:
messages = [
    MemoryMessage(role="user", content="I'm planning a trip to Japan next month!", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Exciting! Based on your preferences, I know you enjoy hiking and vegetarian food. Japan has amazing options for both!", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="user", content="Yes! I'd love to hike Mount Fuji and find good vegetarian ramen.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Perfect! I'll remember your interest in Mount Fuji. For vegetarian ramen, Kyoto has excellent options.", created_at=datetime.now(timezone.utc))
]

updated_memory = WorkingMemory(
    session_id=SESSION_ID, namespace=NAMESPACE, messages=messages, user_id=USER_ID
)

response = await client.put_working_memory(SESSION_ID, updated_memory)
print(f"Working memory updated with {len(messages)} messages")
print(f"Session: {SESSION_ID}")
print(response)

Working memory updated with 4 messages
Session: nitin-travel-session
messages=[MemoryMessage(role='user', content="I'm planning a trip to Japan next month!", id='01KN2MKNVES96JZY8VEF66FAQ7', created_at=datetime.datetime(2026, 3, 31, 19, 6, 0, 430094, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content='Exciting! Based on your preferences, I know you enjoy hiking and vegetarian food. Japan has amazing options for both!', id='01KN2MKNVES96JZY8VEF66FAQ8', created_at=datetime.datetime(2026, 3, 31, 19, 6, 0, 430187, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='user', content="Yes! I'd love to hike Mount Fuji and find good vegetarian ramen.", id='01KN2MKNVES96JZY8VEF66FAQ9', created_at=datetime.datetime(2026, 3, 31, 19, 6, 0, 430214, tzinfo=TzInfo(0)), persisted_at=None, discrete_memory_extracted='f'), MemoryMessage(role='assistant', content="Perfect! I'll remember your interest in Mount Fu

Let's search and see what's actually stored in Redis

In [9]:
# Search all memories for this user
all_memories = await client.search_long_term_memory(
    text="travel preferences",  # Broad search
    namespace={"eq": "travel_agent"},
    user_id={"eq": "nitin"},
    limit=20,
)

print(f"\nFound {all_memories.total} memories:\n")

for idx, memory in enumerate(all_memories.memories, 1):
    # Calculate relevance score (1 - distance)
    relevance = (1 - memory.dist) * 100 if memory.dist else 0

    print(f"{idx}. [{relevance:.0f}% relevant]")
    print(f"   ID: {memory.id}")
    print(f"   Text: {memory.text}")
    print(f"   Topics: {memory.topics}")
    print(f"   Created: {memory.created_at}\n")


Found 5 memories:

1. [48% relevant]
   ID: 01KN01PF648TR7SHJJRZK1QZYV
   Text: Comfort travel - middle tier (not luxury, not budget)
   Topics: ['travel', 'preferences']
   Created: 2026-03-30 18:57:00.100000+00:00

2. [41% relevant]
   ID: 01KN01PF648TR7SHJJRZK1QZYX
   Text: Extra leg room on flights (premium economy or exit row and budget allows, anything goes)
   Topics: ['travel', 'preferences']
   Created: 2026-03-30 18:57:00.100000+00:00

3. [37% relevant]
   ID: 01KN01PF648TR7SHJJRZK1QZYY
   Text: Prefers hotels with good amenities
   Topics: ['travel', 'preferences']
   Created: 2026-03-30 18:57:00.100000+00:00

4. [31% relevant]
   ID: 01KN01PF648TR7SHJJRZK1QZYZ
   Text: Enjoys technology, sports, outdoors, and innovation hubs
   Topics: ['travel', 'preferences']
   Created: 2026-03-30 18:57:00.100000+00:00

5. [25% relevant]
   ID: 01KN01PF648TR7SHJJRZK1QZYW
   Text: Vegetarian diet - needs vegetarian restaurant options
   Topics: ['travel', 'preferences']
   Created: 2026-

# Pattern 2: Background Extraction

Now that you've seen how to manually store and retrieve memories let's look at the real strength of the Redis Agent Memory, automatic memory extraction.

This is the fastest way to show AMS learning from conversations without manual memory creation.

Just store conversations - system extracts memories automatically


### Step 1: Create session with extraction strategy configured

In [10]:
from agent_memory_client.models import MemoryStrategyConfig


SESSION_ID_AUTO = "shannon-auto-session"
USER_ID_AUTO = "shannon"

created, working_memory = await client.get_or_create_working_memory(
    session_id=SESSION_ID_AUTO,
    namespace=NAMESPACE,
    user_id=USER_ID_AUTO,  # Include user_id for consistent session key
    # Configure automatic extraction
    long_term_memory_strategy=MemoryStrategyConfig(
        strategy="discrete"  # Extract individual facts (default)
    ),
)

print(f"Session created with automatic extraction: {SESSION_ID_AUTO}")
strategy = working_memory.long_term_memory_strategy
print(f"Strategy: {strategy.strategy if strategy else 'default'}")

Session created with automatic extraction: shannon-auto-session
Strategy: discrete


### Step 2: Just store the conversation - extraction happens automatically!

In [11]:
conversation = [
    MemoryMessage(role="user", content="I'm Shannon. I'm planning a hiking trip to Japan and need vegetarian food options.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Great choice! Japan has amazing hiking trails and excellent vegetarian cuisine.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="user", content="I prefer nice hotels with good amenities, not too fancy but comfortable. All depends on the budget.", created_at=datetime.now(timezone.utc)),
    MemoryMessage(role="assistant", content="Noted! I'll remember your preference for comfortable mid-tier accommodations.", created_at=datetime.now(timezone.utc))
]

working_memory_update = WorkingMemory(
    session_id=SESSION_ID_AUTO,
    namespace=NAMESPACE,
    messages=conversation,
    user_id=USER_ID_AUTO,
    # Strategy is already configured on the session
)

response = await client.put_working_memory(SESSION_ID_AUTO, working_memory_update)

In [12]:
# pass messages to OpenAI/Claude without additional processing.
prompt_result = await client.memory_prompt(
    session_id=SESSION_ID_AUTO,
    query=user_query,
    namespace=NAMESPACE,
    user_id=USER_ID_AUTO,
    long_term_search={
        "limit": 5,
        # distance_threshold: Lower = stricter when set. If omitted, the server
        # uses no distance filter (distance_threshold=None) for broader KNN recall.
        "user_id": {"eq": USER_ID_AUTO}  # Only search Nitin's memories
    }
)

messages = prompt_result.get("messages", [])

long_term_memories = prompt_result.get("long_term_memories") or []

print(f"Long-term memories found: {memory_count}")

# Display the messages
for msg in messages:
    role = msg.get("role", "unknown")
    content = msg.get("content", {})
    if isinstance(content, dict):
        text = content.get("text", str(content))
    else:
        text = str(content)
    print(f"[{role.upper()}]")
    print(text)
    print()


Long-term memories found: 5
[USER]
I'm Shannon. I'm planning a hiking trip to Japan and need vegetarian food options.

[ASSISTANT]
Great choice! Japan has amazing hiking trails and excellent vegetarian cuisine.

[USER]
I prefer nice hotels with good amenities, not too fancy but comfortable. All depends on the budget.

[ASSISTANT]
Noted! I'll remember your preference for comfortable mid-tier accommodations.

[SYSTEM]
## Long term memories related to the user's query
 No relevant long-term memories found.

[USER]
What are my travel preferences?



### Background Extraction Summary

**Where memory operations happen:**
- `get_or_create_working_memory()` configures extraction strategy
- `put_working_memory()` stores the conversation
- the background worker extracts and stores long-term memories

## Inspect Working Memory

These cells make the session-level state visible so you can show what AMS is holding right now.


In [13]:
# Working Memory Operations

# List all sessions in the travel_agent namespace
sessions = await client.list_sessions(namespace="travel_agent", limit=10)
print(f"Found {sessions.total} session(s) in 'travel_agent' namespace:")
for session in sessions.sessions[:5]:
    print(f"  - {session}")

Found 2 session(s) in 'travel_agent' namespace:
  - nitin-travel-session
  - shannon-auto-session


In [14]:
# Get working memory for a session
# Note: Using the session from Pattern 1
session_to_check = "nitin-travel-session"

_, working_memory = await client.get_or_create_working_memory(
    session_id=session_to_check, namespace="travel_agent"
)
print(f"Session: {working_memory.session_id}")
print(f"Messages: {len(working_memory.messages)}")
print(f"Memories: {len(working_memory.memories)}")
if working_memory.context:
    print(f"Context: {working_memory.context}")

Session: nitin-travel-session
Messages: 4
Memories: 0


In [15]:
# Get working memory for a session
# Note: Using the session from Pattern 1
session_to_check = "shannon-auto-session"

_, working_memory = await client.get_or_create_working_memory(
    session_id=session_to_check, namespace="travel_agent"
)
print(f"Session: {working_memory.session_id}")
print(f"Messages: {len(working_memory.messages)}")
print(f"Memories: {len(working_memory.memories)}")
if working_memory.context:
    print(f"Context: {working_memory.context}")

Session: shannon-auto-session
Messages: 4
Memories: 0


## Search Long-Term Memory

Use these cells to show that promoted memories are searchable and can be filtered or ranked differently.


In [16]:
# Search long-term memories with semantic search

results = await client.search_long_term_memory(
    text="What are Shannon's travel preferences?",
    namespace={"eq": "travel_agent"},
    user_id={"eq": "nitin"},
    limit=5,
)

print(f"Found {results.total} memories for user 'shannon':")
for memory in results.memories:
    relevance = (1 - memory.dist) * 100
    print(f"\n  [{relevance:.0f}%] {memory.text}")
    print(f"       Topics: {memory.topics}")

Found 5 memories for user 'shannon':

  [45%] Comfort travel - middle tier (not luxury, not budget)
       Topics: ['travel', 'preferences']

  [43%] Prefers hotels with good amenities
       Topics: ['travel', 'preferences']

  [34%] Enjoys technology, sports, outdoors, and innovation hubs
       Topics: ['travel', 'preferences']

  [30%] Extra leg room on flights (premium economy or exit row and budget allows, anything goes)
       Topics: ['travel', 'preferences']

  [19%] Vegetarian diet - needs vegetarian restaurant options
       Topics: ['travel', 'preferences']


### Search Modes: Semantic vs Keyword vs Hybrid

You do not need to cover every search knob live, but this cell is useful if you want one quick comparison of search behavior.


In [17]:
# Search Mode Comparison Demo
from agent_memory_client.models import SearchModeEnum

# Semantic search (default) - finds conceptually related memories
semantic_results = await client.search_long_term_memory(
    text="vacation",
    namespace={"eq": "travel_agent"},
    search_mode=SearchModeEnum.SEMANTIC,  # or just "semantic"
    limit=3
)
print(f"SEMANTIC search for 'vacation' ({semantic_results.total} results):")
for mem in semantic_results.memories:
    print(f"  [{mem.score:.3f}] {mem.text[:60]}...")

# Keyword search - exact term matching
keyword_results = await client.search_long_term_memory(
    text="vegetarian",
    namespace={"eq": "travel_agent"},
    search_mode="keyword",
    limit=3
)
print(f"\nKEYWORD search for 'vegetarian' ({keyword_results.total} results):")
for mem in keyword_results.memories:
    print(f"  [{mem.score:.3f}] {mem.text[:60]}...")

# Hybrid search - combines keyword + semantic with configurable weighting
hybrid_results = await client.search_long_term_memory(
    text="vegetarian food options",
    namespace={"eq": "travel_agent"},
    search_mode="hybrid",
    hybrid_alpha=0.7,  # 0.7 = 70% semantic, 30% keyword weight
    limit=3
)
print(f"\nHYBRID search for 'vegetarian food options' ({hybrid_results.total} results):")
for mem in hybrid_results.memories:
    print(f"  [{mem.score:.3f}] {mem.text[:60]}...")

SEMANTIC search for 'vacation' (3 results):
  [0.283] Comfort travel - middle tier (not luxury, not budget)...
  [0.215] Prefers hotels with good amenities...
  [0.211] Extra leg room on flights (premium economy or exit row and b...

KEYWORD search for 'vegetarian' (1 results):
  [1.000] Vegetarian diet - needs vegetarian restaurant options...

HYBRID search for 'vegetarian food options' (3 results):
  [1.000] Vegetarian diet - needs vegetarian restaurant options...
  [0.443] Prefers hotels with good amenities...
  [0.440] Comfort travel - middle tier (not luxury, not budget)...


## One More Example: Episodic Memory

This optional section is a good closer if you want to show that AMS handles more than static preferences.


In [18]:
# Create an episodic memory with event date
from datetime import UTC, timedelta


episodic_memory = ClientMemoryRecord(
    text="User completed a 10-mile hike at Rocky Mountain National Park",
    memory_type=MemoryTypeEnum.EPISODIC,
    topics=["activities", "hiking", "achievements"],
    entities=["Rocky Mountain National Park"],
    event_date=datetime.now(UTC) - timedelta(days=7),
    namespace=NAMESPACE,
)

result = await client.create_long_term_memory([episodic_memory])
print(f"Created episodic memory: {result.status}")
print(f"Event date: {episodic_memory.event_date}")

Created episodic memory: ok
Event date: 2026-03-24 19:06:02.338253+00:00


## Cleanup

Optional: remove demo sessions after the walkthrough.


In [19]:
# Cleanup demo data (optional)
cleanup = True  # Set to True to delete demo data
from agent_memory_client.models import ForgetPolicy

if cleanup:
    sessions_to_delete = await client.list_sessions()
    for sid in sessions_to_delete.sessions:
        try:
            await client.delete_working_memory(sid, namespace="travel_agent")

            await client.forget_long_term_memories(
                policy=ForgetPolicy(budget=0),
                session_id="my-session",
                dry_run=False,
                limit=1000,
            )
            print(f"Deleted session: {sid}")
        except Exception:
            print(f"Error deleting session: {sid}")
            pass
else:
    print("Cleanup skipped. Set cleanup=True to delete demo data.")

Deleted session: nitin-travel-session
Deleted session: shannon-auto-session


## Summary

Recommended 25-minute flow:

1. Health check and client setup
2. Code-driven memory creation and `memory_prompt()`
3. Working-memory write and inspection
4. Background extraction
5. Long-term search
6. Optional episodic-memory example

In [ ]:
sessions = await client.list_sessions()
print(sessions.sessions)

[]
